In [1]:
import faiss
import numpy as np
import json

# Load embeddings and metadata
embeddings = np.load("perfume_vectors.npy")
with open("perfume_metadata.json", "r") as f:
    metadata = json.load(f)

print("Embeddings shape:", embeddings.shape)
print("Example:", metadata[0])


Embeddings shape: (24063, 1671)
Example: {'Perfume': 'accento-overdose-pride-edition', 'url': 'https://www.fragrantica.com/perfume/xerjoff/accento-overdose-pride-edition-74630.html'}


In [2]:
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

In [3]:
# define dimensionality
dim = embeddings.shape[1]

# create index
index = faiss.IndexFlatIP(dim)

# add embeddings
index.add(embeddings)

print("Total perfumes indexed:", index.ntotal)

Total perfumes indexed: 24063


In [5]:
#Testing

query = np.zeros(dim, dtype=np.float32)

note_to_idx = {note: i for i, note in enumerate(open("note_vocab.txt").read().splitlines())}

for note, confidence in {"jasmine":0.993, "sage":0.963, "amber":0.942, "ambergris":0.838}.items():
    if note in note_to_idx:
        query[note_to_idx[note]] = confidence

#normalize the query vector
query = query / np.linalg.norm(query)

#search top 5 similar perfumes
distances, indices = index.search(query.reshape(1,-1), k = 5)

for rank, (idx, score) in enumerate(zip(indices[0], distances[0]), 1):
    name = metadata[idx]["Perfume"]
    url = metadata[idx]["url"]
    print(f"{rank}. {name} ({score:.3f}) — {url}")

1. amalia-primavera (0.677) — https://www.fragrantica.com/perfume/fueguia-1833/amalia-primavera-18239.html
2. soft-and-young (0.669) — https://www.fragrantica.com/perfume/verset-parfums/soft-and-young-59213.html
3. white-gardenia (0.669) — https://www.fragrantica.com/perfume/zara/white-gardenia-68681.html
4. white-luminous-gold (0.669) — https://www.fragrantica.com/perfume/michael-kors/white-luminous-gold-31343.html
5. mysore-incenza (0.623) — https://www.fragrantica.com/perfume/areej-le-dore/mysore-incenza-76652.html


In [6]:
faiss.write_index(index, "perfume_index.faiss")